In [1]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Load UCI promoter dataset directly
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/molecular-biology/promoter-gene-sequences/promoters.data"

df = pd.read_csv(url, header=None, names=['label', 'id', 'sequence'])

In [2]:
def clean_sequence(seq):
    seq = seq.upper()
    seq = seq.strip()

    return seq
df_cleaned = df.copy()
df_cleaned['label'] = df_cleaned['label'].map({'-': 0, '+': 1})
df_cleaned['sequence'] = df_cleaned['sequence'].apply(clean_sequence)
df_cleaned.head(3)

,label,id,sequence
0,1,S10,TACTAGCAATACGCTTGCGTTCGGTGGTTAAGTATGTATAATGCGC...
1,1,AMPC,TGCTATCCTGACAGTTGTCACGCTGATTGGTGTCGTTACAATCTAA...
2,1,AROH,GTACTAGAGAACTAGTGCATTAGCTTATTTTTTTGTTATCATGCTA...


In [3]:
class PromoterDataset(Dataset):
    def __init__(self, sequences, labels, encoder):
        self.sequences = sequences
        self.labels = labels
        self.encoder = encoder
        self.encoded_sequences = [self.encoder.encode(seq).T for seq in self.sequences]
        self.encoded_labels = torch.tensor(self.labels, dtype=torch.float32)
        self.encoded_sequences = torch.stack(self.encoded_sequences) # Transpose to (4,57)
    
    def __len__(self):
        return len(self.encoded_sequences)
    
    def __getitem__(self, idx):
        return self.encoded_sequences[idx], self.encoded_labels[idx]

In [12]:
import wandb
import nbformat
import yaml

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

torch.manual_seed(config['training']['random_seed'])
np.random.seed(config['training']['random_seed'])
# Start a run
wandb.init(
    project="promoter-cnn-classifier",
    name="batchnorm_experiment",
    config=config
)

In [13]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import sys
sys.path.append('../src/')
from UpdatedSequenceEncoder import SequenceEncoder
from promoter_cnn import PromoterCNNClassifier

# Prepare data
sequences = df_cleaned['sequence'].values
labels = df_cleaned['label'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies = []
fold_losses = []

# Initialize best tracking
best_accuracy = 0
best_fold_idx = -1
best_model_state = None

for fold, (train_idx, test_idx) in enumerate(skf.split(sequences, labels)):
    # 1. Split data using train_idx, test_idx
    train_sequences, train_labels = sequences[train_idx], labels[train_idx]
    test_sequences, test_labels = sequences[test_idx], labels[test_idx]
    
    # 2. Create PromoterDataset for each split
    train_dataset = PromoterDataset(train_sequences, train_labels, SequenceEncoder())
    test_dataset = PromoterDataset(test_sequences, test_labels, SequenceEncoder())

    # 3. Create DataLoaders (batch_size=8)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

    # 4. Create FRESH model + optimizer + criterion

    model = PromoterCNNClassifier(4, wandb.config['model']['hidden_size'], 1)
    criterion = torch.nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=wandb.config['training']['lr'])
    # 5. Train for 50 epochs
    for epoch in range(wandb.config['training']['epochs']):
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            y_batch = y_batch.float()
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs.squeeze(), y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss
        wandb.log({
    'train_loss': epoch_loss / len(train_loader)
}, step=epoch + (fold * wandb.config['training']['epochs']))

    # 6. Evaluate on test fold
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        final_loss = 0
        for X_batch, y_batch in test_loader:
            predictions = model(X_batch)
            predicted_labels = (predictions > 0.5).float()
            correct += (predicted_labels.squeeze() == y_batch).sum().item()
            total += len(y_batch)
            final_loss += criterion(predictions.squeeze(), y_batch.float()).item()
    # 7. Append accuracy to fold_accuracies
    fold_losses.append(final_loss / len(test_loader))        
    accuracy = correct / total if total > 0 else 0
    fold_accuracies.append(accuracy)
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_fold_idx = fold
        best_model_state = model.state_dict()
    wandb.log({
    'fold': fold + 1,
    'fold_accuracy': accuracy,
})
wandb.run.summary["mean_accuracy"] = np.mean(fold_accuracies)
wandb.run.summary["std_accuracy"] = np.std(fold_accuracies)
wandb.run.summary["best_fold"] = best_fold_idx + 1
wandb.run.summary["best_fold_accuracy"] = best_accuracy
# Save best model
torch.save(best_model_state, f"model_fold{best_fold_idx + 1}.pth")
# Log as artifact
model_artifact = wandb.Artifact(f"promoter-cnn-fold{best_fold_idx + 1}", type="model")
model_artifact.add_file(f"model_fold{best_fold_idx + 1}.pth")
wandb.log_artifact(model_artifact)
wandb.finish()
# If train loss << test loss → overfitting
# If train loss ≈ test loss → good generalization

fold,▁▃▅▆█
fold_accuracy,█▆██▁
train_loss,▁▁▁▂▁▁▁▄▁▃▁▁▂▂▁▂▂▂▁▂▁▁▁▁█▂▁▁▁▁▁▃▂▁▁▁▁▁▁▁
best_fold,1
best_fold_accuracy,1
fold,5
fold_accuracy,0.85714
mean_accuracy,0.9619
std_accuracy,0.05553
train_loss,0.00321
